# Amazon Review Alignment: T4 Smoke

使用 `Qwen/Qwen3-0.6B` 在 Colab T4 上执行：

`Base -> SFT -> DPO -> Reward Model -> PPO -> GRPO -> Evaluation`

所有训练阶段都是极小规模 smoke，只验证链路与产物，不代表收敛。


## 1. 挂载 Drive 并加载项目

本 Notebook 将仓库放在 Google Drive，训练输出和 checkpoint 会在断开
Colab 后保留。目标 GPU：`T4`。

开始前必须先将本地最新代码和本 Notebook 提交并推送到 GitHub `main`
分支，否则下方 `git clone` 会获取旧版本。


In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/j156734119/Amazon-reviews-2023-electronics-SFT-DPO.git"
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--short"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        print("Preserving Colab-local tracked changes before pull:")
        print(status)
        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "stash",
                "push",
                "-m",
                "colab-auto-stash-before-pull",
            ],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


## 2. 安装依赖

执行后使用 Colab 菜单 **运行时 -> 重新启动会话**。重启后从下一单元格
继续，不需要再次执行安装。


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,eval,dev]"],
    check=True,
)
print("Installation complete. Restart the Colab runtime now.")


## 3. 重启后恢复目录、加载 Secrets

在 Colab 左侧钥匙图标中添加 `OPENAI_API_KEY`。`HF_TOKEN` 对公开模型
是可选的，但能提高 Hugging Face 下载限额。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata
from packaging.version import Version

drive.mount("/content/drive", force_remount=False)
REPO_DIR = Path(
    "/content/drive/MyDrive/amazon-review-alignment-workspace/repo"
)
os.chdir(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
CONFIG = "configs/rlhf_smoke.yaml"

def training_stack_is_usable() -> bool:
    try:
        import amazon_review_alignment
        import bitsandbytes
        import peft
        import transformers
        import trl
    except (ImportError, ModuleNotFoundError):
        return False
    return Version(bitsandbytes.__version__) >= Version("0.46.1")

if not training_stack_is_usable():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            ".[train,eval,dev]",
        ],
        cwd=REPO_DIR,
        check=True,
    )

try:
    import amazon_review_alignment
    import bitsandbytes
    import peft
    import transformers
    import trl
except (ImportError, ModuleNotFoundError) as exc:
    raise RuntimeError(
        "Training dependencies are still unavailable after reinstall. "
        "Restart the Colab runtime and rerun this cell."
    ) from exc

for secret_name in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[secret_name] = value

def cli(*arguments: str, check: bool = True) -> subprocess.CompletedProcess:
    command = [
        sys.executable,
        "-m",
        "amazon_review_alignment.cli",
        *(str(argument) for argument in arguments),
    ]
    print("\n$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(
        command,
        returncode,
        stdout="".join(output_lines),
        stderr=None,
    )
    if check and returncode:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: "
            + " ".join(command)
        )
    return result

print("Config:", CONFIG)
print("Package:", Path(amazon_review_alignment.__file__).resolve())
print(
    "Training stack:",
    transformers.__version__,
    trl.__version__,
    peft.__version__,
    bitsandbytes.__version__,
)
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("HF token loaded:", bool(os.getenv("HF_TOKEN")))


## T4 环境检查


In [ ]:
import importlib.metadata

import torch
import transformers
import trl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a Colab GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"VRAM: {total_gib:.2f} GiB")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", importlib.metadata.version("peft"))

if "T4".lower() not in gpu_name.lower():
    raise RuntimeError("Expected T4, but Colab assigned: " + gpu_name)
if total_gib < 14:
    raise RuntimeError("Insufficient GPU memory for this profile.")
if False and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This profile requires BF16 support.")


## 4. 测试、准备数据并生成 Base baseline


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest"],
    cwd=REPO_DIR,
    check=True,
)
cli("prepare-data", "--config", CONFIG)


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "--force-inference",
)

import pandas as pd
from amazon_review_alignment.config import load_config

output_root = Path(
    load_config(CONFIG)["project"]["output_dir"]
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))


## 5. 教师数据

Pilot 会立即调用 OpenAI API。Batch 提交后可能需要等待；提交成功后可以
关闭 GPU Runtime，稍后重新连接并重复“检查 Batch”单元格。


In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets first.")
cli("teacher-pilot", "--config", CONFIG)


In [ ]:
# 第一次执行会提交 Batch；后续重复执行会查询并下载结果。
cli("teacher-batch", "--config", CONFIG)


In [ ]:
from amazon_review_alignment.config import load_config

output_root = Path(load_config(CONFIG)["project"]["output_dir"])
train_preferences = output_root / "teacher" / "preferences_train.jsonl"
validation_preferences = (
    output_root / "teacher" / "preferences_validation.jsonl"
)
if not train_preferences.exists() or not validation_preferences.exists():
    raise RuntimeError(
        "Batch is not complete. Re-run the previous cell later; "
        "do not start training yet."
    )
print("Teacher train rows:", sum(1 for _ in train_preferences.open()))
print(
    "Teacher validation rows:",
    sum(1 for _ in validation_preferences.open()),
)


## 6. SFT、合并权重与 DPO


In [ ]:
cli("train-sft", "--config", CONFIG)


In [ ]:
cli("merge-sft", "--config", CONFIG)


In [ ]:
cli("train-dpo", "--config", CONFIG)


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
    "--force-inference",
)
display(pd.read_csv(output_root / "evaluation" / "metrics.csv"))


## 7. Smoke 人工偏好校准

对 4 组回答逐条输入 `A`、`B` 或 `tie`。至少保留两条非 tie 记录。


In [ ]:
cli("prepare-rm-human-eval", "--config", CONFIG, "--samples", "4")

import pandas as pd

responses_path = output_root / "rlhf" / "rm_human_responses.csv"
frame = pd.read_csv(responses_path, keep_default_na=False)

for index, row in frame.iterrows():
    print("\n" + "=" * 80)
    print("Review:", row["review_text"])
    print("\nA:", row["response_a"])
    print("\nB:", row["response_b"])
    while True:
        choice = input("选择 A / B / tie: ").strip().lower()
        if choice in {"a", "b", "tie"}:
            break
    frame.at[index, "choice"] = choice

frame.to_csv(responses_path, index=False)
print("Saved:", responses_path)


In [ ]:
cli(
    "build-rlhf-data",
    "--config",
    CONFIG,
    "--responses",
    str(output_root / "rlhf" / "rm_human_responses.csv"),
)


## 8. Reward Model、PPO 与 GRPO

每个阶段是独立单元格。阶段失败时先处理报错，不要跳过并继续。


In [ ]:
cli("train-reward", "--config", CONFIG)


In [ ]:
cli("train-ppo", "--config", CONFIG)


In [ ]:
cli("train-grpo", "--config", CONFIG)


## 9. 五模型统一评估和报告


In [ ]:
cli(
    "evaluate",
    "--config",
    CONFIG,
    "--variants",
    "base",
    "sft",
    "dpo",
    "ppo",
    "grpo",
)
cli("build-report", "--config", CONFIG)

metrics_path = output_root / "evaluation" / "metrics.csv"
report_path = output_root / "evaluation" / "report.md"
display(pd.read_csv(metrics_path))
print("Report:", report_path.resolve())
